# Attendance Grading

* Step 1: Download an updated version of the gradebook
* Step 2: Input the proper session length into the session_length variable
* Step 3: Input the csv names into the respective variables
* Step 4: Input the assignment name from the gradebook into the assignment_name variable
* Step 5: Make sure the zoom recording csv has records occuring after the due date removed before running results.

Results are retrieved in "grading_results.csv". The remaining csv's are used for debugging purposes. "Testing.csv" returns with more columns joined to the gradebook: "Name", "Duration (minutes)" "View Duration (minutes)", "Total View Time (minutes) and "Grade" to verify that the grading script is placing the correct grades in the correct student's row. "Testing.csv" is for debugging/testing purposes not to be put on Canvas.

## CSV's used for testing/debugging purposes
* Preliminary.csv
* Testing.csv
* grade_count.csv
* live_and_recording.csv
* grade_verification.csv

## CSV's required for script to function
* Gradebook csv (gradebook_name)
* Zoom recording csv (recording_csv_name)
* Zoom live csv (live_csv_name)

## Other necessary inputs
* Zoom session length (session_length)
* Name of assignment in gradebook (assignment_name)

Final results are placed in the "grading_results.csv" use that for inputting into canvas.

TODO: Removing columns from the zoom recording csv where attendance is inadmissible for credit (student watched too late) automatically rather than having to manually remove them.

In [ ]:
import pandas as pd

# Set the variable session_length to how long the zoom session was for to accurately get grading results (in minutes)
session_length = 186

# Name of attendance csv (change as needed)
recording_csv_name = "zoomus_recordinganalytics_01-27-2025-play.csv"
live_csv_name = "zoomus_live_participants_01-27-2025.csv"

# Name of assignment in the gradebook (change as needed)
assignment_name = "1/27/25 Attendance (2257195)"

# Gradebook name (change as needed)
gradebook_name = "2025-02-10T1622_Grades-COT5930_012_16128.csv"

# Date where attendance viewing is no longer counted. Format in mm-dd-yyyy
# cutoff_date = "01-06-2025"

In [ ]:
# # Find the date info from the recording name and extract the month date and year from the csv filename

# mat = re.findall("[0-9]", csv_name)

# month_date_year = []
# for i in range(0, len(mat) - 4, 2):
#   month_date_year.append(''.join(mat[i:i+2]))
#   # mat = ''.join(mat[:2])
# month_date_year.append(''.join(mat[-4:]))

# # Input date time parameters
# beginning_month = int(month_date_year[0][1:])
# beginning_date = int(month_date_year[1][1:])
# beginning_year = int(month_date_year[2])

# # Input the end date parameters here
# end_month = 1
# end_date = 14
# end_year = 2025

# beginning_period = datetime.datetime(beginning_year, beginning_month, beginning_date)
# end_period = datetime.datetime(end_year, end_month, end_date)

# print(end_period)
# print(beginning_period)



In [ ]:
# def create_datetime(end_period, row_date):



#   dt = datetime.datetime.strptime(row_date, "%y%y")
#   print(dt)
#   # month_date_year = []
#   # for i in range(0, len(mat) - 4, 2):
#   #   month_date_year.append(''.join(mat[i:i+2]))
#   # # mat = ''.join(mat[:2])
#   # month_date_year.append(''.join(mat[-4:]))

#   # # Input date time parameters
#   # month = int(month_date_year[0][1:])
#   # date = int(month_date_year[1][1:])
#   # year = int(month_date_year[2])


#   dt = datetime.datetime(year, month, date)

#   return dt <= end_period

In [ ]:
# Read initial zoom csv recording and live
live_df = pd.read_csv(live_csv_name)
recording_df = pd.read_csv(recording_csv_name)
recording_df = recording_df.rename(columns={"User Email": "Email"})


# Fill in 0's for "< 1" minute in recording and live csv's 
live_df['Duration (minutes)'] = live_df['Duration (minutes)'].apply(lambda x: 0 if x == '< 1' else int(x))
recording_df["View Duration (minutes)"] = recording_df["View Duration (minutes)"].apply(lambda x: 0 if x == '< 1' else int(x))

# Sum up the student viewtimes for live and recording respectively
live_df = live_df.groupby(["Email"], as_index=False)["Duration (minutes)"].sum()
recording_df = recording_df.groupby(["Email"], as_index=False)["View Duration (minutes)"].sum()

# Concat both live and recording dataframes together
frames = [live_df, recording_df]
live_and_recording_df = pd.concat(frames)

# Group students by their email and sum up their
live_and_recording_df = live_and_recording_df.groupby(['Email']).agg({'Duration (minutes)': 'sum', 'View Duration (minutes)': 'sum'})

# Reset email from being the index of the dataframe
live_and_recording_df = live_and_recording_df.reset_index()


live_and_recording_df["Total View Time (minutes)"] = live_and_recording_df["Duration (minutes)"] + live_and_recording_df["View Duration (minutes)"]
live_and_recording_df.to_csv("./Debugging_csv's/live_and_recording.csv")
live_and_recording_df



In [ ]:
# Changing Total view time column to int dtype
int_dict = {'Total View Time (minutes)': int}
live_and_recording_df = live_and_recording_df.astype(int_dict)
print(live_and_recording_df.dtypes)

In [ ]:
# Implementing grading scale function

def grading_scale(session_length, df):

  ninety_percent_watch = round(session_length * 0.9)
  eighty_percent_watch = round(session_length * 0.8)
  sixty_percent_watch = round(session_length * 0.6)
  forty_percent_watch = round(session_length * 0.4)
  thirty_percent_watch = round(session_length * 0.3)

  df['Grade'] = 'NA'
  df.loc[(df['Total View Time (minutes)'] < thirty_percent_watch), 'Grade'] = 0

  df.loc[(df['Total View Time (minutes)'] >= thirty_percent_watch) & (df['Total View Time (minutes)'] < forty_percent_watch), 'Grade'] = 1

  df.loc[(df['Total View Time (minutes)'] >= forty_percent_watch) & (df['Total View Time (minutes)'] < sixty_percent_watch), 'Grade'] = 2

  df.loc[(df['Total View Time (minutes)'] >= sixty_percent_watch) & (df['Total View Time (minutes)'] < eighty_percent_watch), 'Grade'] = 3

  df.loc[(df['Total View Time (minutes)'] >= eighty_percent_watch) & (df['Total View Time (minutes)'] < ninety_percent_watch), 'Grade'] = 4

  df.loc[(df['Total View Time (minutes)'] >= ninety_percent_watch), 'Grade'] = 5

In [ ]:
# Call grading scale with live student attendance, recording student attendance, and session length csv

grading_scale(session_length, live_and_recording_df)

# Used in testing whether the grades were accurately applied to students in the gradebook
live_and_recording_df.to_csv("./Debugging_csv's/Preliminary.csv")

In [ ]:
# Read in the course gradebook
course_gradebook_df = pd.read_csv(gradebook_name)

# Change zoom recording column name from email to SIS Login ID for joining with gradebook csv
live_and_recording_df = live_and_recording_df.rename(columns={"Email": "SIS Login ID"})

live_and_recording_df_final = live_and_recording_df.copy()

# Left join the gradebook with the live and recording dataframe (experimental is for debugging)
merged_df_experimental = pd.merge(course_gradebook_df, live_and_recording_df, on="SIS Login ID", how="left")
merged_df_final = pd.merge(course_gradebook_df, live_and_recording_df_final, on="SIS Login ID", how="left")

# Create dataframe with students names, emails, view times, and grade for debugging 
grade_verification_df = pd.concat([merged_df_experimental[["Student", assignment_name]], merged_df_experimental[["SIS Login ID", "Duration (minutes)", "View Duration (minutes)", "Total View Time (minutes)", "Grade"]]], axis=1)
grade_verification_df.to_csv("./Debugging_csv's/grade_verification.csv")

# Move Attendance grades into appropriate column at appropriate rows
merged_df_experimental.loc[2:, assignment_name] = merged_df_experimental.loc[2:, "Grade"]
merged_df_final.loc[2:, assignment_name] = merged_df_final.loc[2:, "Grade"]

# Drop unwanted columns from final dataframe
merged_df_final = merged_df_final.drop(["Duration (minutes)", "View Duration (minutes)", "Total View Time (minutes)", "Grade"], axis=1)

# Fill in missing values with 0's (missing values means they did not watch live or recording at all)
merged_df_experimental = merged_df_experimental.fillna(value = {assignment_name: 0})
merged_df_experimental.loc[2:] = merged_df_experimental.loc[2:].fillna(value = {"Grade": 0})
merged_df_final = merged_df_final.fillna(value = {assignment_name: 0})

# CSV where watch times, names, and grades are included to verify the script grades properly
merged_df_experimental.to_csv("./Debugging_csv's/Testing.csv", index=False)

# Final gradebook to submit to Canvas
merged_df_final.to_csv('grading_results.csv', index=False)



In [ ]:
# Gives grade counts for the class (Not yet complete)
grade_count = merged_df_experimental.copy()
grade_count = grade_count.groupby(['Grade'])['Grade'].size()
grade_count.to_csv('grade_count.csv')

In [ ]:
# Used to compare the gradebook df with the merged_df_final to see if it only changed the assignment name column in the gradebook

column_list = list(course_gradebook_df.columns)
column_list.remove(assignment_name)


comparison_df = merged_df_final.merge(course_gradebook_df, on=column_list, how='outer', suffixes=['', '_'], indicator=True)

comparison_df.to_csv("./Debugging_csv's/comparison.csv")
